In [2]:
import os

import numpy as np
import pandas as pd

In [3]:
session_dir = "/tabletop/bags/session_2025-08-12_15-19-30"
# loaded_dfs = rosbag_session_to_dfs(
#     session_dir, topics=["/eyelink/sample", "/markers"], save=True
# )
# eyelink_loaded_df = loaded_dfs["/eyelink/sample"]
# markers_loaded_df = loaded_dfs["/markers"]

eyelink_csv_path = os.path.join(session_dir, "eyelink_sample.csv")
eyelink_received_csv_path = os.path.join(
    session_dir, "eyelink_received", "last.csv"
)
markers_csv_path = os.path.join(session_dir, "markers.csv")

eyelink_df = pd.read_csv(eyelink_csv_path)
eyelink_received_df = pd.read_csv(eyelink_received_csv_path)
markers_df = pd.read_csv(markers_csv_path)
marker_count = 1

eyelink_df.rename(
    columns={
        "time": "recorded_time",
        "timestamp.sec": "header.stamp.sec",
        "timestamp.nanosec": "header.stamp.nanosec",
    },
    inplace=True,
)
column_map = {
    f"markers[{i}].translation.{axis}": f"marker_{i}_{axis}"
    for i in range(marker_count)
    for axis in ["x", "y", "z"]
}
markers_df.rename(
    columns={
        "time": "recorded_time",
        **column_map,
    },
    inplace=True,
)

In [4]:
print(eyelink_df.head())

         recorded_time  header.stamp.sec  header.stamp.nanosec  eyelink_time  \
0  1755011971486194149        1755011971             486194149    1276845029   
1  1755011971486591600        1755011971             486591600    1276845030   
2  1755011971487443786        1755011971             487443786    1276845031   
3  1755011971488427534        1755011971             488427534    1276845032   
4  1755011971489443930        1755011971             489443930    1276845033   

    left_x   left_y  left_pupil  right_x  right_y  right_pupil  input  
0 -32768.0 -32768.0         0.0  -2521.0 -13074.0       3965.0    247  
1 -32768.0 -32768.0         0.0  -2500.0 -13050.0       3955.0    247  
2 -32768.0 -32768.0         0.0  -2475.0 -13048.0       3958.0    247  
3 -32768.0 -32768.0         0.0  -2475.0 -13048.0       3961.0    247  
4 -32768.0 -32768.0         0.0  -2476.0 -13044.0       3961.0    247  


In [5]:
markers_df

,recorded_time,header.stamp.sec,header.stamp.nanosec,header.frame_id,header_original.stamp.sec,header_original.stamp.nanosec,header_original.frame_id,frame_number,markers[0].id_type,markers[0].marker_index,markers[0].marker_name,marker_0_x,marker_0_y,marker_0_z
0,1755011970869292488,1755011970,868433384,map,1755011970,869217668,map,6463312,1.0,0.0,NaN,-0.272010,0.172933,0.322673
1,1755011970877506300,1755011970,876806844,map,1755011970,877452904,map,6463313,1.0,0.0,NaN,-0.273700,0.171240,0.324085
2,1755011970889268174,1755011970,885129598,map,1755011970,885893426,map,6463314,1.0,0.0,NaN,-0.275326,0.169582,0.325482
3,1755011970894688238,1755011970,893699525,map,1755011970,894672933,map,6463315,1.0,0.0,NaN,-0.276966,0.167968,0.327042
4,1755011970902931567,1755011970,901930098,map,1755011970,902894902,map,6463316,1.0,0.0,NaN,-0.278624,0.166744,0.329223
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7434,1755012032815761520,1755012032,815263133,map,1755012032,815703825,map,6470746,1.0,0.0,NaN,-0.130409,0.098330,0.457387
7435,1755012032824099563,1755012032,823566777,map,1755012032,824052901,map,6470747,1.0,0.0,NaN,-0.130537,0.097774,0.458862
7436,1755012032832330430,1755012032,831905352,map,1755012032,832304604,map,6470748,1.0,0.0,NaN,-0.130683,0.097231,0.460269
7437,1755012032840845690,1755012032,840220828,map,1755012032,840808512,map,6470749,1.0,0.0,NaN,-0.130842,0.096705,0.461686


In [64]:
edf = eyelink_df.copy(deep=True)
mdf = markers_df.copy(deep=True)

edf["time"] = edf["header.stamp.sec"] + edf["header.stamp.nanosec"] / 1e9
mdf["time"] = mdf["header.stamp.sec"] + mdf["header.stamp.nanosec"] / 1e9

edf = edf[
    [
        "time",
        "left_x",
        "left_y",
        "left_pupil",
        "right_x",
        "right_y",
        "right_pupil",
    ]
]
mdf = mdf[
    [
        "time",
        *[
            f"marker_{i}_{ax}"
            for i in range(marker_count)
            for ax in ["x", "y", "z"]
        ],
    ]
]

# edf.set_index("time", inplace=True)
# mdf.set_index("time", inplace=True)

assert edf.index.is_monotonic_increasing
assert mdf.index.is_monotonic_increasing

In [65]:
edf

,time,left_x,left_y,left_pupil,right_x,right_y,right_pupil
0,1.755012e+09,-32768.0,-32768.0,0.0,-2521.0,-13074.0,3965.0
1,1.755012e+09,-32768.0,-32768.0,0.0,-2500.0,-13050.0,3955.0
2,1.755012e+09,-32768.0,-32768.0,0.0,-2475.0,-13048.0,3958.0
3,1.755012e+09,-32768.0,-32768.0,0.0,-2475.0,-13048.0,3961.0
4,1.755012e+09,-32768.0,-32768.0,0.0,-2476.0,-13044.0,3961.0
...,...,...,...,...,...,...,...
61309,1.755012e+09,-3921.0,-11560.0,4975.0,-4099.0,-10796.0,4129.0
61310,1.755012e+09,-3903.0,-11560.0,4974.0,-4102.0,-10790.0,4130.0
61311,1.755012e+09,-3903.0,-11560.0,4974.0,-4102.0,-10791.0,4130.0
61312,1.755012e+09,-3903.0,-11562.0,4972.0,-4089.0,-10792.0,4123.0


In [66]:
mdf

,time,marker_0_x,marker_0_y,marker_0_z
0,1.755012e+09,-0.272010,0.172933,0.322673
1,1.755012e+09,-0.273700,0.171240,0.324085
2,1.755012e+09,-0.275326,0.169582,0.325482
3,1.755012e+09,-0.276966,0.167968,0.327042
4,1.755012e+09,-0.278624,0.166744,0.329223
...,...,...,...,...
7434,1.755012e+09,-0.130409,0.098330,0.457387
7435,1.755012e+09,-0.130537,0.097774,0.458862
7436,1.755012e+09,-0.130683,0.097231,0.460269
7437,1.755012e+09,-0.130842,0.096705,0.461686


In [67]:
# edf = edf.drop(index=pd.RangeIndex(start=10000, stop=11000))

In [68]:
print(
    f"Eyelink time_diff min: {edf['time'].diff().min():.4f}, max: {edf['time'].diff().max():.4f}, expected: {1 / 1000:.4f}"
)
print(
    f"Markers time_diff min: {mdf['time'].diff().min():.4f}, max: {mdf['time'].diff().max():.4f}, expected: {1 / 120:.4f}"
)

Eyelink time_diff min: 0.0004, max: 0.0016, expected: 0.0010
Markers time_diff min: 0.0065, max: 0.0100, expected: 0.0083


In [69]:
merged_df = pd.merge_asof(
    edf,
    mdf,
    on="time",
    direction="backward",
    tolerance=0.001,
)
merged_df

,time,left_x,left_y,left_pupil,right_x,right_y,right_pupil,marker_0_x,marker_0_y,marker_0_z
0,1.755012e+09,-32768.0,-32768.0,0.0,-2521.0,-13074.0,3965.0,NaN,NaN,NaN
1,1.755012e+09,-32768.0,-32768.0,0.0,-2500.0,-13050.0,3955.0,NaN,NaN,NaN
2,1.755012e+09,-32768.0,-32768.0,0.0,-2475.0,-13048.0,3958.0,NaN,NaN,NaN
3,1.755012e+09,-32768.0,-32768.0,0.0,-2475.0,-13048.0,3961.0,NaN,NaN,NaN
4,1.755012e+09,-32768.0,-32768.0,0.0,-2476.0,-13044.0,3961.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
61309,1.755012e+09,-3921.0,-11560.0,4975.0,-4099.0,-10796.0,4129.0,NaN,NaN,NaN
61310,1.755012e+09,-3903.0,-11560.0,4974.0,-4102.0,-10790.0,4130.0,NaN,NaN,NaN
61311,1.755012e+09,-3903.0,-11560.0,4974.0,-4102.0,-10791.0,4130.0,NaN,NaN,NaN
61312,1.755012e+09,-3903.0,-11562.0,4972.0,-4089.0,-10792.0,4123.0,-0.130441,0.099496,0.454273


In [70]:
print(f"Num matched: {merged_df.marker_0_x.notna().sum()}")
print(f"Expected num unmatched: {mdf.shape[0]}")

Num matched: 7302
Expected num unmatched: 7439


In [ ]:
asof = merged_df.asof(merged_df.index)
asof["time"].diff() > (1/)

0.7090682983398438

In [38]:
asof.isna().sum(axis=0)

time           8
left_x         8
left_y         8
left_pupil     8
right_x        8
right_y        8
right_pupil    8
marker_0_x     8
marker_0_y     8
marker_0_z     8
dtype: int64

In [27]:
def interp(df, new_index):
    """Return a new DataFrame with all columns values interpolated
    to the new_index values."""
    df_out = pd.DataFrame(index=new_index)
    df_out.index.name = df.index.name

    for name in df.columns:
        df_out[name] = np.interp(new_index, df.index, df[name])

    return df_out


mdf_interp = interp(mdf, edf.index)
mdf_interp

,time,marker_0_x,marker_0_y,marker_0_z
0,1.755012e+09,-0.272010,0.172933,0.322673
1,1.755012e+09,-0.273700,0.171240,0.324085
2,1.755012e+09,-0.275326,0.169582,0.325482
3,1.755012e+09,-0.276966,0.167968,0.327042
4,1.755012e+09,-0.278624,0.166744,0.329223
...,...,...,...,...
61309,1.755012e+09,-0.131046,0.096204,0.463062
61310,1.755012e+09,-0.131046,0.096204,0.463062
61311,1.755012e+09,-0.131046,0.096204,0.463062
61312,1.755012e+09,-0.131046,0.096204,0.463062


In [10]:
merged_df = pd.merge(edf, mdf_interp, on="time")
merged_df

,time,left_x,left_y,left_pupil,right_x,right_y,right_pupil,marker_0_x,marker_0_y,marker_0_z
0,1.755012e+09,-7350.0,-11076.0,4400.0,-7569.0,-10155.0,3599.0,-0.173129,0.191348,0.328784
1,1.755012e+09,-4048.0,-12333.0,4177.0,-4196.0,-11802.0,3645.0,-0.178593,0.097098,0.460632
2,1.755012e+09,-6394.0,-11971.0,4335.0,-6334.0,-11175.0,3701.0,-0.157734,0.101374,0.365314
3,1.755012e+09,-989.0,-8365.0,4206.0,-1155.0,-8265.0,3622.0,-0.054357,0.168631,0.572862
4,1.755012e+09,-1783.0,-8082.0,3871.0,-2387.0,-8204.0,3253.0,-0.194999,0.280431,0.517783


In [ ]:
merged_df = pd.merge(eyelink_df, markers_df, on="time")

merged_df.to_csv(os.path.join(session_dir, "merged.csv"), index=False)